# ENERGIZE NILM — Unstructured Pruning

Runs global **L1 unstructured pruning** over a list of sparsity levels.

For each sparsity level the notebook:
1. Reloads the trained baseline checkpoint
2. Applies `torch.nn.utils.prune.global_unstructured` (L1Unstructured)
3. Evaluates the pruned model (before fine-tuning)
4. Fine-tunes for `FINETUNE_EPOCHS` epochs (masks kept active → zeroed weights remain zero)
5. Removes pruning masks (bakes zeros into weights) and saves checkpoint
6. Evaluates the fine-tuned model

**Works with CNN, GRU, and TCN** — unlike structured pruning which cannot handle GRU/LSTM layers.

> Unstructured pruning does not change the model shape or tensor sizes,
> so MACs and parameter counts remain identical to the baseline.
> The compression metric is **sparsity %** (fraction of zero weights).

In [ ]:
# ============================================================================
# COLAB SETUP — run this cell first
# ============================================================================
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from tqdm import tqdm

from src_pytorch import (
    CNN_NILM, TCN_NILM, GRU_NILM,
    SimpleNILMDataLoader,
    set_seeds, get_device,
    compute_status, compute_metrics,
    get_model_config, get_appliance_params,
    get_model_stats, get_model_sparsity,
    apply_unstructured_pruning, remove_pruning_masks,
    get_prunable_parameters,
)

## Configuration

Edit **only this cell**.

In [ ]:
# ============================================================================
# USER CONFIGURATION — edit only this cell
# ============================================================================

FINETUNE_EPOCHS = 10      # fine-tuning epochs after each pruning level
FINETUNE_LR     = 1e-4   # Adam learning rate for fine-tuning
SEED            = 42

# Experiments: list of dicts with model, appliance, and sparsity levels
# Sparsity = fraction of weights to zero (0.50 → 50% of weights become 0)
EXPERIMENTS = [
    {
        'model'           : 'tcn',
        'appliance'       : 'boiler',
        'sparsity_levels' : [0.25, 0.50, 0.75, 0.85, 0.90, 0.95, 0.99],
    },
    # Uncomment to add more experiments:
    # {'model': 'cnn', 'appliance': 'boiler',          'sparsity_levels': [0.25, 0.50, 0.75, 0.90, 0.95]},
    # {'model': 'gru', 'appliance': 'boiler',          'sparsity_levels': [0.25, 0.50, 0.75, 0.90, 0.95]},
    # {'model': 'tcn', 'appliance': 'washing_machine', 'sparsity_levels': [0.25, 0.50, 0.75, 0.90, 0.95]},
]

DATASET = 'plegma'
# ============================================================================

In [ ]:
# ============================================================================
# AUTO-DERIVED — do not edit
# ============================================================================
NOTEBOOK_DIR = Path('.').resolve()
REPO_ROOT    = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
DATA_DIR     = REPO_ROOT / 'data'
OUTPUTS_DIR  = REPO_ROOT / 'outputs'

set_seeds(SEED)
DEVICE = get_device()

print(f'Repo root : {REPO_ROOT}')
print(f'Data dir  : {DATA_DIR}')
print(f'Device    : {DEVICE}')

## Helper Functions

In [ ]:
def build_model(model_name: str, cfg: dict, ckpt_path, device):
    """Instantiate and load a fresh model from a checkpoint."""
    ckpt_path = Path(ckpt_path)
    if not ckpt_path.exists():
        raise FileNotFoundError(
            f'Checkpoint not found: {ckpt_path}\n'
            'Run 01_data_prep_training.ipynb first.'
        )
    window = cfg['input_window_length']
    if model_name == 'cnn':
        model = CNN_NILM(input_window_length=window)
    elif model_name == 'gru':
        model = GRU_NILM(input_window_length=window)
    elif model_name == 'tcn':
        model = TCN_NILM(
            input_window_length=window,
            depth=cfg.get('depth', 9),
            nb_filters=cfg.get('nb_filters'),
            dropout=cfg.get('dropout', 0.2),
            stacks=cfg.get('stacks', 1),
        )
    else:
        raise ValueError(f'Unknown model: {model_name}')
    model.load_state_dict(torch.load(str(ckpt_path), map_location=device))
    return model.to(device).eval()


def evaluate(model, data_loader, model_name, app_params, cfg, device):
    """Run inference on the test split and return metrics + predictions."""
    from src_pytorch.evaluator import evaluate_model

    return evaluate_model(
        model=model,
        data_loader=data_loader,
        model_name=model_name,
        cutoff=app_params['cutoff'],
        threshold=app_params['threshold'],
        device=device,
        input_window_length=cfg['input_window_length'],
        min_on=app_params.get('min_on'),
        min_off=app_params.get('min_off'),
        max_length=app_params.get('max_length'),
    )


def finetune(model, data_loader, model_name, epochs, lr, device):
    """Fine-tune model in-place. Pruning masks (if present) stay active."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()
    model.train()
    for epoch in range(1, epochs + 1):
        total_loss = 0.0
        n_batches  = 0
        for batch_x, batch_y in tqdm(data_loader.train, desc=f'  Epoch {epoch}/{epochs}'):
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_x)
            if model_name in ('cnn', 'gru') and batch_y.dim() == 1:
                batch_y = batch_y.unsqueeze(1)
            elif model_name == 'tcn' and batch_y.dim() == 2:
                batch_y = batch_y.unsqueeze(-1)
            loss = loss_fn(outputs, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches  += 1
        print(f'  Epoch {epoch}/{epochs} — avg MSE: {total_loss / n_batches:.6f}')
    model.eval()


def save_predictions(output_dir, slug, gt, pred, gt_status, pred_status):
    pred_dir = output_dir / 'predictions'
    pred_dir.mkdir(parents=True, exist_ok=True)
    path = pred_dir / f'{slug}_preds.csv'
    np.savetxt(
        path,
        np.column_stack([gt, pred, gt_status.astype(int), pred_status.astype(int)]),
        delimiter=',',
        header='ground_truth_W,prediction_W,ground_truth_status,predicted_status',
        comments='', fmt=['%.4f', '%.4f', '%d', '%d'],
    )


def print_metrics(metrics, label=''):
    print(f'  {"-"*40}')
    if label:
        print(f'  {label}')
    print(f'  MAE        : {metrics["mae"]:.4f} W')
    print(f'  F1         : {metrics["f1"]:.4f}')
    if 'f1_complex' in metrics:
        print(f'  F1 Complex : {metrics["f1_complex"]:.4f}')
    print(f'  Accuracy   : {metrics["accuracy"]:.4f}')
    print(f'  Precision  : {metrics["precision"]:.4f}')
    print(f'  Recall     : {metrics["recall"]:.4f}')
    print(f'  {"-"*40}')

## Define `run_experiment`

In [ ]:
def run_experiment(model_name: str, appliance_name: str, sparsity_levels: list):
    """
    Full unstructured pruning pipeline for one (model, appliance) pair.

    For each sparsity level:
      1. Reload baseline checkpoint
      2. Apply global L1 unstructured pruning
      3. Evaluate pruned (no finetune)
      4. Fine-tune (masks kept active during training)
      5. Remove masks → save checkpoint
      6. Evaluate fine-tuned

    Returns a dict with baseline and per-level results.
    """
    print(f'\n{"#"*70}')
    print(f'  EXPERIMENT: {model_name.upper()} | {appliance_name.upper()}')
    print(f'{"#"*70}')

    cfg        = get_model_config(model_name)
    app_params = get_appliance_params(DATASET, appliance_name)
    output_dir = OUTPUTS_DIR / f'{model_name}_{appliance_name}'
    models_dir = output_dir / 'models'
    models_dir.mkdir(parents=True, exist_ok=True)

    ckpt_path = output_dir / 'checkpoint' / 'model.pt'
    label     = f'{model_name}_{appliance_name}'

    # ── Data loader ──────────────────────────────────────────────────────────
    data_loader = SimpleNILMDataLoader(
        data_dir=str(DATA_DIR / DATASET / appliance_name),
        model_name=model_name,
        batch_size=cfg['batch_size'],
        input_window_length=cfg['input_window_length'],
        train=True,
        num_workers=0,
    )

    # ── Baseline evaluation ───────────────────────────────────────────────────
    print(f'\n{"-"*60}')
    print(f'  Baseline evaluation')
    print(f'{"-"*60}')
    base_model = build_model(model_name, cfg, ckpt_path, DEVICE)

    dummy    = torch.randn(1, cfg['input_window_length']).to(DEVICE)
    b_params, b_macs, b_mb = get_model_stats(base_model, dummy)

    print(f'  Params: {b_params:,}  |  MACs: {b_macs:,}  |  Size: {b_mb:.3f} MB')
    print(f'  Prunable parameters:')
    for m, n in get_prunable_parameters(base_model):
        w = getattr(m, n)
        print(f'    {type(m).__name__}.{n:30s} {list(w.shape)}')

    base_metrics, b_gt, b_pred, b_gt_s, b_pred_s = evaluate(
        base_model, data_loader, model_name, app_params, cfg, DEVICE
    )
    print_metrics(base_metrics, label='Baseline')
    save_predictions(output_dir, f'{label}_unstructured_baseline', b_gt, b_pred, b_gt_s, b_pred_s)

    all_results = []

    # ── Pruning loop ──────────────────────────────────────────────────────────
    for sparsity in sparsity_levels:
        pct = int(sparsity * 100)
        print(f'\n{"="*60}')
        print(f'  Sparsity {pct}%  |  {model_name.upper()}  |  {appliance_name}')
        print(f'{"="*60}')

        # Reload fresh baseline weights
        model = build_model(model_name, cfg, ckpt_path, DEVICE)

        # ── 1. Apply unstructured pruning ────────────────────────────────────
        model = apply_unstructured_pruning(model, amount=sparsity)
        actual_sparsity = get_model_sparsity(model)

        # ── 2. Evaluate pruned (no finetune) ─────────────────────────────────
        pruned_metrics, p_gt, p_pred, p_gt_s, p_pred_s = evaluate(
            model, data_loader, model_name, app_params, cfg, DEVICE
        )
        print_metrics(pruned_metrics, label=f'Pruned {pct}% (no finetune)')
        save_predictions(output_dir, f'{label}_unstructured_pruned_{pct}pct', p_gt, p_pred, p_gt_s, p_pred_s)

        # ── 3. Fine-tune (masks kept active) ─────────────────────────────────
        print(f'\n  Fine-tuning {FINETUNE_EPOCHS} epochs  |  LR={FINETUNE_LR}')
        finetune(model, data_loader, model_name, FINETUNE_EPOCHS, FINETUNE_LR, DEVICE)

        # ── 4. Remove masks → save checkpoint ────────────────────────────────
        remove_pruning_masks(model)
        ckpt_ft = models_dir / f'{label}_unstructured_{pct}pct_finetuned.pt'
        torch.save(model.state_dict(), ckpt_ft)
        print(f'  Checkpoint saved: {ckpt_ft.name}')

        # ── 5. Evaluate fine-tuned ────────────────────────────────────────────
        ft_metrics, ft_gt, ft_pred, ft_gt_s, ft_pred_s = evaluate(
            model, data_loader, model_name, app_params, cfg, DEVICE
        )
        print_metrics(ft_metrics, label=f'Pruned {pct}% + Finetuned {FINETUNE_EPOCHS}ep')
        save_predictions(output_dir, f'{label}_unstructured_{pct}pct_finetuned', ft_gt, ft_pred, ft_gt_s, ft_pred_s)

        all_results.append({
            'sparsity_target'  : pct,
            'sparsity_actual'  : round(actual_sparsity * 100, 2),
            'params'           : b_params,
            'macs'             : b_macs,
            'mb'               : b_mb,
            'pruned_metrics'   : pruned_metrics,
            'finetuned_metrics': ft_metrics,
        })

    return {
        'model'           : model_name,
        'appliance'       : appliance_name,
        'baseline_metrics': base_metrics,
        'baseline_params' : b_params,
        'baseline_macs'   : b_macs,
        'baseline_mb'     : b_mb,
        'all_results'     : all_results,
        'output_dir'      : output_dir,
    }

## Run All Experiments

In [ ]:
all_experiments = []

for _exp in EXPERIMENTS:
    _result = run_experiment(
        model_name     = _exp['model'],
        appliance_name = _exp['appliance'],
        sparsity_levels= _exp['sparsity_levels'],
    )
    all_experiments.append(_result)

print(f'\n{"#"*70}')
print(f'  All {len(EXPERIMENTS)} experiment(s) complete.')
print(f'{"#"*70}')

## Summary Tables & Save Results

In [ ]:
def _v(d, k, fmt='.4f'):
    v = d.get(k)
    return f'{v:{fmt}}' if v is not None else '—'


for exp in all_experiments:
    m_name  = exp['model'].upper()
    app     = exp['appliance'].replace('_', ' ').title()
    bm      = exp['baseline_metrics']

    print(f'\n{"="*80}')
    print(f'  {m_name}  |  {app}  —  Unstructured Pruning Results')
    print(f'{"="*80}')

    rows = []

    # Baseline row
    rows.append({
        'Stage'          : f'{m_name} Baseline',
        'Sparsity_%'     : 0,
        'Params'         : exp['baseline_params'],
        'MACs'           : exp['baseline_macs'],
        'MB'             : exp['baseline_mb'],
        'MAE'            : round(bm['mae'],                  4),
        'F1'             : round(bm['f1'],                   4),
        'F1_Complex'     : round(bm['f1_complex'], 4) if bm.get('f1_complex') is not None else None,
        'Precision'      : round(bm['precision'],            4),
        'Recall'         : round(bm['recall'],               4),
        'Accuracy'       : round(bm['accuracy'],             4),
        'Energy_Error_%' : round(bm['energy_error_percent'], 2),
    })

    for r in exp['all_results']:
        pct = r['sparsity_target']
        for stage_key, stage_label in [
            ('pruned_metrics',    f'{m_name} Pruned {pct}%'),
            ('finetuned_metrics', f'{m_name} Pruned {pct}% + Finetuned {FINETUNE_EPOCHS}ep'),
        ]:
            sm = r[stage_key]
            rows.append({
                'Stage'          : stage_label,
                'Sparsity_%'     : r['sparsity_actual'],
                'Params'         : r['params'],
                'MACs'           : r['macs'],
                'MB'             : r['mb'],
                'MAE'            : round(sm['mae'],                  4),
                'F1'             : round(sm['f1'],                   4),
                'F1_Complex'     : round(sm['f1_complex'], 4) if sm.get('f1_complex') is not None else None,
                'Precision'      : round(sm['precision'],            4),
                'Recall'         : round(sm['recall'],               4),
                'Accuracy'       : round(sm['accuracy'],             4),
                'Energy_Error_%' : round(sm['energy_error_percent'], 2),
            })

    df = pd.DataFrame(rows)
    display(df)

    # ── Save to Excel ─────────────────────────────────────────────────────────
    xlsx_path = exp['output_dir'] / f'{exp["model"]}_{exp["appliance"]}_unstructured_results.xlsx'
    df.to_excel(xlsx_path, index=False)
    print(f'  Results saved -> {xlsx_path.relative_to(REPO_ROOT)}')

## Plot — F1 Complex vs Sparsity

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

C_NAVY   = '#1B3F7A'
C_ORANGE = '#C55A11'

for exp in all_experiments:
    m_name  = exp['model'].upper()
    app     = exp['appliance'].replace('_', ' ').title()
    bm      = exp['baseline_metrics']
    results = exp['all_results']

    sparsity_pts = [0] + [r['sparsity_actual'] for r in results]
    f1_pruned    = [bm.get('f1_complex', bm['f1'])] + [
        r['pruned_metrics'].get('f1_complex', r['pruned_metrics']['f1']) for r in results
    ]
    f1_finetuned = [bm.get('f1_complex', bm['f1'])] + [
        r['finetuned_metrics'].get('f1_complex', r['finetuned_metrics']['f1']) for r in results
    ]

    fig, ax = plt.subplots(figsize=(10, 6))

    ax.plot(sparsity_pts, f1_pruned,
            color=C_ORANGE, marker='o', linewidth=1.5, markersize=7,
            linestyle=':', alpha=0.8,
            label=f'{m_name} Unstructured (no finetune)')

    ax.plot(sparsity_pts, f1_finetuned,
            color=C_NAVY, marker='D', linewidth=2.2, markersize=8,
            linestyle='--',
            label=f'{m_name} Unstructured + Finetuned')

    ax.set_xlabel('Sparsity (%)', fontsize=12)
    ax.set_ylabel('F1 Score (Complex)', fontsize=12)
    ax.set_title(
        f'{app}: Sparsity (%) vs F1 Complex  [{m_name} Unstructured]',
        fontsize=14, fontweight='bold', pad=12
    )
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%g%%'))
    ax.grid(True, linestyle='--', alpha=0.35)
    ax.legend(fontsize=10, loc='lower left', framealpha=0.9)
    plt.tight_layout()

    fig_dir  = exp['output_dir'] / 'figures'
    fig_dir.mkdir(exist_ok=True)
    fig_path = fig_dir / f'{exp["model"]}_{exp["appliance"]}_unstructured_f1complex_vs_sparsity.png'
    fig.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f'Figure saved -> {fig_path.relative_to(REPO_ROOT)}')
    plt.show()